In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 3, Finished, Available, Finished, False)

In [2]:
df_historical = spark.read.option("header",True).csv("Files/Ahmedabad_Historical_Data.csv")
display(df_historical)

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 31c683b8-621d-4c70-9742-0f8ceac1022c)

In [3]:
df_daily = spark.read.option("multiline",True).json("Files/bronze/*.json")

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 6, Finished, Available, Finished, False)

In [4]:
df_market = (
    df_daily
    .select(explode("records").alias("record"))
    .select("record.*")
)

display(df_market)

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 773686f4-d2ff-4a12-b9f2-43dd46524c36)

In [5]:
def standardize_schema(df):
    return (
        df.select(
            to_date(col("Arrival_Date"), "dd/MM/yyyy").alias("arrival_date"),
            trim(col("Commodity")).alias("commodity"),
            col("Commodity_Code").cast("int").alias("commodity_code"),
            trim(initcap(col("District"))).alias("district"),
            trim(col("Grade")).alias("grade"),
            trim(initcap(col("Market"))).alias("market"),
            col("Max_Price").cast("double").alias("max_price"),
            col("Min_Price").cast("double").alias("min_price"),
            col("Modal_Price").cast("double").alias("modal_price"),
            trim(initcap(col("State"))).alias("state"),
            trim(col("Variety")).alias("variety")
        )
    )

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 8, Finished, Available, Finished, False)

In [6]:
df_historical_clean = standardize_schema(df_historical)
df_daily_clean = standardize_schema(df_market)

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 9, Finished, Available, Finished, False)

In [7]:
df_combined = df_historical_clean.unionByName(df_daily_clean)
print("Combined rows before dedup:", df_combined.count())

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 10, Finished, Available, Finished, False)

Combined rows before dedup: 58658


In [8]:
df_combined.show(3)

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 11, Finished, Available, Finished, False)

+------------+---------+--------------+---------+-----+---------+---------+---------+-----------+-------+-------+
|arrival_date|commodity|commodity_code| district|grade|   market|max_price|min_price|modal_price|  state|variety|
+------------+---------+--------------+---------+-----+---------+---------+---------+-----------+-------+-------+
|  2024-07-08|  Cabbage|           154|Ahmedabad|  FAQ|Ahmedabad|   3300.0|   1200.0|     2950.0|Gujarat|  Other|
|  2024-07-10|  Cabbage|           154|Ahmedabad|  FAQ|Ahmedabad|   3800.0|   1500.0|     3050.0|Gujarat|  Other|
|  2024-07-13|  Cabbage|           154|Ahmedabad|  FAQ|Ahmedabad|   3800.0|   2000.0|     3100.0|Gujarat|  Other|
+------------+---------+--------------+---------+-----+---------+---------+---------+-----------+-------+-------+
only showing top 3 rows



In [9]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy(
    "state", "district", "market", "commodity", "variety", "arrival_date"
).orderBy(col("modal_price").desc())

df_deduped = (
    df_combined
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

print("Rows after dedup:", df_deduped.count())

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 13, Finished, Available, Finished, False)

Rows after dedup: 58535


In [29]:
for lh in notebookutils.lakehouse.list():
    print(lh)

StatementMeta(, e3de5851-0ec7-4650-ac3b-ab255d10947e, 40, Finished, Available, Finished, False)

{'id': '39e67300-4542-423f-b716-14916f69a745', 'type': 'Lakehouse', 'displayName': 'lh_rawcsvdata', 'description': '', 'workspaceId': '19509fa9-5985-4a20-a114-28961871f698', 'properties': {'abfsPath': 'abfss://19509fa9-5985-4a20-a114-28961871f698@onelake.dfs.fabric.microsoft.com/39e67300-4542-423f-b716-14916f69a745'}}
{'id': '65a2f421-9db7-4ec8-8fa6-39387d60b013', 'type': 'Lakehouse', 'displayName': 'lh_silver', 'description': '', 'workspaceId': '19509fa9-5985-4a20-a114-28961871f698', 'properties': {'abfsPath': 'abfss://19509fa9-5985-4a20-a114-28961871f698@onelake.dfs.fabric.microsoft.com/65a2f421-9db7-4ec8-8fa6-39387d60b013'}}


In [10]:
silver_path = (
    "abfss://19509fa9-5985-4a20-a114-28961871f698@onelake.dfs.fabric.microsoft.com/"
    "65a2f421-9db7-4ec8-8fa6-39387d60b013/Files/silver_mandi_prices"
)

df_deduped.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 14, Finished, Available, Finished, False)

In [11]:
df_deduped.write \
    .format("delta") \
    .mode("overwrite") \
    .save("Files/silver_mandi_prices")

StatementMeta(, faad12ec-7a84-4bf6-8dcb-dc9425067dac, 15, Finished, Available, Finished, False)